# 第 54 课｜Numba 数值循环加速与验算

本 Notebook 对应「365 天量化金融」第 54 课。课程方法来自本地完整视频、课件截图和本地 Whisper 口播整理；这里把它重建为可运行、可验证的量化工程教学实现。

- 图文口播：`/Users/mojiawen/Documents/学习资料归档/_暂存/pam-video-captures/365-days-quant-finance/54_youtube-fad02SOf_Zw/index.html`
- 环境：`pkm-quant`
- 数据：固定随机种子的 NumPy 收益数组
- 状态：可运行学习稿；不代表生产交易系统

## 本节量化问题

Numba 把适合的数值循环即时编译成机器码。正确基准要把首次编译时间与预热后的运行时间分开，并用 Python 参考实现检查结果一致。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


In [2]:
# Numba 已进入统一 pkm-quant 环境。
import numba
from numba import njit
import time
print("Numba:", numba.__version__)

Numba: 0.66.0


## 1. 一个逐时点回撤循环

输入是 `float64` 净值数组，输出每个时点相对历史峰值的回撤。该算法有顺序依赖，不适合简单 `prange`，但很适合 `njit` 编译显式循环。

In [3]:
# Python 参考实现和 Numba 实现保持完全相同的控制流。
def drawdown_python(nav):
    output = np.empty_like(nav)
    peak = nav[0]
    for i in range(len(nav)):
        if nav[i] > peak:
            peak = nav[i]
        output[i] = nav[i] / peak - 1.0
    return output

@njit
def drawdown_numba(nav):
    output = np.empty_like(nav)
    peak = nav[0]
    for i in range(len(nav)):
        if nav[i] > peak:
            peak = nav[i]
        output[i] = nav[i] / peak - 1.0
    return output

small_nav = np.array([1.00, 1.10, 1.05, 1.20, 1.08])
expected = np.array([0.0, 0.0, 1.05 / 1.10 - 1, 0.0, 1.08 / 1.20 - 1])
np.testing.assert_allclose(drawdown_python(small_nav), expected)
np.testing.assert_allclose(drawdown_numba(small_nav), expected)
display(pd.DataFrame({"nav": small_nav, "drawdown": expected}))

,nav,drawdown
0,1.00,0.000000
1,1.10,0.000000
2,1.05,-0.045455
3,1.20,0.000000
4,1.08,-0.100000


## 2. 首次调用 = 编译 + 运行

第一次输入某种 dtype 时，Numba 要生成机器码；后续同类型调用才接近稳定运行时间。不能用首次 Numba 时间与已预热的 Python/NumPy 时间比较。

In [4]:
# 分别记录首次编译调用和预热后的重复运行。
rng = np.random.default_rng(365)
returns = rng.normal(0.0002, 0.01, 500_000)
nav = np.cumprod(1 + returns).astype(np.float64)

start = time.perf_counter()
first_result = drawdown_numba(nav)
first_time = time.perf_counter() - start

start = time.perf_counter()
warmed_result = drawdown_numba(nav)
warmed_time = time.perf_counter() - start

start = time.perf_counter()
python_result = drawdown_python(nav)
python_time = time.perf_counter() - start

print(f"Numba 首次: {first_time:.4f}s")
print(f"Numba 预热后: {warmed_time:.4f}s")
print(f"Python 循环: {python_time:.4f}s")
print(f"本机预热加速比: {python_time / warmed_time:.1f}x")
np.testing.assert_allclose(first_result, python_result, rtol=1e-12, atol=1e-12)
np.testing.assert_allclose(warmed_result, python_result, rtol=1e-12, atol=1e-12)

Numba 首次: 0.0007s
Numba 预热后: 0.0007s
Python 循环: 0.0656s
本机预热加速比: 94.0x


## 3. dtype 是编译签名的一部分

`float32` 与 `float64` 会形成不同签名，也可能带来不同舍入误差。性能优化后要明确 `rtol/atol`，不能因为结果“看起来差不多”就跳过验算。

In [5]:
# 展示已编译签名，并用合理容差比较 float32。
float32_result = drawdown_numba(nav.astype(np.float32))
print("已编译签名:")
for signature in drawdown_numba.signatures:
    print(" ", signature)
np.testing.assert_allclose(float32_result, python_result, rtol=1e-5, atol=1e-6)
assert len(drawdown_numba.signatures) >= 2

已编译签名:
  (Array(float64, 1, 'C', False, aligned=True),)
  (Array(float32, 1, 'C', False, aligned=True),)


## 方法边界与复用结论


- Numba 最适合数值型 NumPy 数组和显式循环，不直接适合 Pandas/字符串/动态 Python 对象。
- 向量化 NumPy 已很快的代码未必能从 JIT 获益。
- 课程中的百倍加速不是保证；必须在目标硬件、真实数据量和预热后实测。


本课首先保护数据正确性、时间一致性和可复现性，再讨论速度或自动化。局部代码通过不代表整套交易系统满足权限、监控、灾备和实盘延迟要求。